# 01 — Study area and base spatial data

This notebook builds the broader Sevilla spatial context used by the proof-of-concept.

It downloads:

1. the Sevilla administrative boundary;
2. the pedestrian street network;
3. building footprints from OpenStreetMap.

The outputs are saved once and reused by the later notebooks.

In [ ]:
from pathlib import Path

# The notebooks_v02 folder is expected to sit directly inside the repository.
# If you open Jupyter from another working directory, set PROJECT_ROOT manually.
HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name == "notebooks_v02" else HERE

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
OUTPUTS = PROJECT_ROOT / "outputs"

for folder in [DATA_RAW, DATA_INTERIM, OUTPUTS]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox

PLACE = "Sevilla, Andalucía, Spain"

## 1. Download the administrative boundary

In [ ]:
boundary = ox.geocode_to_gdf(PLACE)

print("Boundary CRS:", boundary.crs)
display(boundary[["display_name", "geometry"]].head())

## 2. Download the walking network

In [ ]:
G = ox.graph_from_place(
    PLACE,
    network_type="walk",
    simplify=True,
)

print("Walking-network nodes:", G.number_of_nodes())
print("Walking-network edges:", G.number_of_edges())

## 3. Download building footprints

In [ ]:
buildings = ox.features_from_place(
    PLACE,
    tags={"building": True},
)

# Keep areal building geometries only.
buildings = buildings[
    buildings.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
].copy()

print("Building footprints:", len(buildings))

## 4. Visual check

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

boundary.boundary.plot(ax=ax, linewidth=1.2)
buildings.plot(ax=ax, linewidth=0, alpha=0.35)

ax.set_title("Sevilla study context and OSM building footprints")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 5. Save reusable base data

In [ ]:
ox.save_graphml(
    G,
    filepath=DATA_RAW / "sevilla_walk.graphml",
)

boundary.to_file(
    DATA_RAW / "sevilla_boundary.gpkg",
    driver="GPKG",
)

buildings.to_file(
    DATA_RAW / "sevilla_buildings.gpkg",
    driver="GPKG",
)

print("Saved raw spatial inputs to:", DATA_RAW)